In [1]:
# Import required libraries
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:
# Database connection configuration to SIMPEG
DB_HOST = os.getenv('DB_HOST_SIMPEG', 'localhost')
DB_PORT = os.getenv('DB_PORT_SIMPEG', '5432')  # PostgreSQL default port
DB_NAME = os.getenv('DB_DATABASE_SIMPEG', 'your_database_name')
DB_USER = os.getenv('DB_USERNAME_SIMPEG', 'your_username')
DB_PASSWORD = os.getenv('DB_PASSWORD_SIMPEG', 'your_password')

# Create connection string for PostgreSQL
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"Connecting to database: {DB_NAME} on {DB_HOST}:{DB_PORT}")
print(f"User: {DB_USER}")

# Test connection
try:
    engine_simpeg = create_engine(connection_string, echo=False)
    with engine_simpeg.connect() as connection:
        result = connection.execute(text("SELECT 1 as test"))
        print("✅ Database connection successful!")
        print(f"Connection test result: {result.fetchone()[0]}")
except SQLAlchemyError as e:
    print(f"❌ Database connection failed: {e}")
    print("Please check your database credentials in the .env file")


Connecting to database: simpeg_jabar on 10.110.32.121:5432
User: postgres
✅ Database connection successful!
Connection test result: 1


In [3]:
# Database connection configuration to TRK 2025
DB_HOST = os.getenv('DB_HOST_2025', 'localhost')
DB_PORT = os.getenv('DB_PORT_2025', '5432')  # PostgreSQL default port
DB_NAME = os.getenv('DB_DATABASE_2025', 'your_database_name')
DB_USER = os.getenv('DB_USERNAME_2025', 'your_username')
DB_PASSWORD = os.getenv('DB_PASSWORD_2025', 'your_password')

# Create connection string for PostgreSQL
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"Connecting to database: {DB_NAME} on {DB_HOST}:{DB_PORT}")
print(f"User: {DB_USER}")

# Test connection
try:
    engine_2025 = create_engine(connection_string, echo=False)
    with engine_2025.connect() as connection:
        result = connection.execute(text("SELECT 1 as test"))
        print("✅ Database connection successful!")
        print(f"Connection test result: {result.fetchone()[0]}")
except SQLAlchemyError as e:
    print(f"❌ Database connection failed: {e}")
    print("Please check your database credentials in the .env file")


Connecting to database: erk_ekinerja_2025 on 10.110.32.114:5432
User: postgres
✅ Database connection successful!
Connection test result: 1


In [4]:
def load_data_from_sql(query, engine):
    """
    Load data from PostgreSQL database into a pandas DataFrame.
    
    Parameters:
    query (str): SQL query to execute
    engine: SQLAlchemy engine object
    
    Returns:
    pandas.DataFrame: Data from the query
    """
    connection = None
    try:
        # Create a new connection and rollback any pending transaction
        connection = engine.connect()
        
        # Rollback any pending transaction to ensure clean state
        try:
            connection.rollback()
        except:
            pass  # If no transaction to rollback, ignore
        
        # Execute the query
        df = pd.read_sql(query, connection)
        print(f"✅ Data loaded successfully! Shape: {df.shape}")
        
        connection.close()
        return df
    except Exception as e:
        # Ensure connection is closed on error
        if connection:
            try:
                connection.rollback()
            except:
                pass
            try:
                connection.close()
            except:
                pass
        print(f"❌ Error loading data: {e}")
        return None

def get_table_info(table_name, engine):
    """
    Get basic information about a table.
    
    Parameters:
    table_name (str): Name of the table
    engine: SQLAlchemy engine object
    """
    try:
        # Get table structure using PostgreSQL information_schema
        structure_query = f"""
            SELECT 
                column_name,
                data_type,
                character_maximum_length,
                is_nullable,
                column_default
            FROM information_schema.columns
            WHERE table_name = '{table_name}'
            ORDER BY ordinal_position
        """
        structure = pd.read_sql(structure_query, engine)
        
        # Get row count
        count_query = f"SELECT COUNT(*) as row_count FROM {table_name}"
        count_result = pd.read_sql(count_query, engine)
        
        print(f"📊 Table: {table_name}")
        print(f"Rows: {count_result['row_count'].iloc[0]}")
        print(f"Columns: {len(structure)}")
        print("\nColumn Information:")
        print(structure)
        
        return structure
    except SQLAlchemyError as e:
        print(f"❌ Error getting table info: {e}")
        return None

def list_tables(engine):
    """
    List all tables in the database.
    
    Parameters:
    engine: SQLAlchemy engine object
    """
    try:
        # Use PostgreSQL information_schema to list tables
        query = """
            SELECT table_name 
            FROM information_schema.tables 
            WHERE table_schema = 'public'
            ORDER BY table_name
        """
        tables = pd.read_sql(query, engine)
        print("📋 Available tables:")
        for table in tables['table_name']:
            print(f"  - {table}")
        return tables
    except SQLAlchemyError as e:
        print(f"❌ Error listing tables: {e}")
        return None

print("Data loading functions defined successfully!")



Data loading functions defined successfully!


In [5]:
# Load the csv

df_validasi_atasan = pd.read_csv('Atasan Validasi SKP.csv')

df_validasi_atasan.head()

,nip_atasan,nip_bawahan
0,197105061998031003,199604022025051005
1,197105061998031003,199706242025051002
2,197608252005011008,200006182025051002
3,197608252005011008,199706192025051003
4,197406172008011006,200008242025052004


In [10]:
# Clean df_validasi_atasan

df_validasi_atasan = df_validasi_atasan.applymap(lambda x: str(x).strip().replace("'", "").replace('"', ""))


In [11]:
# Load vpd pkls

df_vpd_akhir_tahun = pd.read_pickle('df_vpd_akhir_tahun.pkl')
df_vpd_tw4 = pd.read_pickle('df_vpd_tw4.pkl')

In [14]:
df_validasi_atasan[df_validasi_atasan['nip_atasan'].str.contains('PLT')]['nip_atasan'].str.split('-', expand=True)

,0,1
26,PLT,10020737
27,PLT,10020737
28,PLT,10020737
29,PLT,10020737
30,PLT,10020737
...,...,...
289,PLT,11914
290,PLT,11914
291,PLT,11914
292,PLT,11914


In [17]:
def execute_update(query, params, engine, description=""):
    """
    Execute an UPDATE query and return success status.
    
    Parameters:
    query (str): SQL UPDATE query
    params (dict): Parameters for the query
    engine: SQLAlchemy engine
    description (str): Description for logging
    
    Returns:
    dict: {'success': bool, 'rows_affected': int, 'error': str}
    """
    connection = None
    try:
        connection = engine.connect()
        
        # Rollback any pending transaction
        try:
            connection.rollback()
        except:
            pass
        
        # Execute the UPDATE with parameters
        result = connection.execute(text(query), params)
        rows_affected = result.rowcount
        
        # Commit the transaction
        connection.commit()
        connection.close()
        
        return {
            'success': True,
            'rows_affected': rows_affected,
            'error': None,
            'description': description
        }
    except Exception as e:
        # Rollback on error
        if connection:
            try:
                connection.rollback()
            except:
                pass
            try:
                connection.close()
            except:
                pass
        
        return {
            'success': False,
            'rows_affected': 0,
            'error': str(e),
            'description': description
        }

def atasan_validasi_skp(df_val, df_vpdat, df_vpdt, engine, tahunan=True, tw4=True):
    """
    Update atasan validation in SKP databases.
    
    Returns:
    pd.DataFrame: Log of all update operations
    """
    # Prepare log
    log = []
    
    for index, row in df_val.iterrows():
        nip_bawahan = row['nip_bawahan']
        nip_atasan = row['nip_atasan']
        
        # If bawahan PLT
        if 'PLT' in str(nip_bawahan):
            try:
                jabatan_id = nip_bawahan.split('-')[1]

                # Check jabatan_id or tugas_tambahan_jabatan_id from vpd (using both tw4 and tahunan)
                peg_jabatan_tw4 = df_vpdt[df_vpdt['jabatan_id'] == jabatan_id]
                peg_jabatan_tahunan = df_vpdat[df_vpdat['jabatan_id'] == jabatan_id]
                peg_jabatan = pd.concat([peg_jabatan_tw4, peg_jabatan_tahunan], ignore_index=True)

                peg_tugas_tambahan_tw4 = df_vpdt[df_vpdt['tugas_tambahan_jabatan_id'] == jabatan_id]
                peg_tugas_tambahan_tahunan = df_vpdat[df_vpdat['tugas_tambahan_jabatan_id'] == jabatan_id]
                peg_tugas_tambahan = pd.concat([peg_tugas_tambahan_tw4, peg_tugas_tambahan_tahunan], ignore_index=True)

                if not peg_jabatan.empty:
                    peg_id = peg_jabatan['peg_id'].iloc[0]
                    if tw4:
                        query = """
                        UPDATE v_pegawai_data_tw4_2025
                        SET peg_jabatan = null
                        WHERE peg_id = :peg_id
                        """
                        result = execute_update(query, {'peg_id': peg_id}, engine, f"Update vpd_tw4 for {nip_bawahan}")
                        log.append({
                            'nip_bawahan': nip_bawahan,
                            'nip_atasan': nip_atasan,
                            'table': 'v_pegawai_data_tw4_2025',
                            'status': 'success' if result['success'] else 'failed',
                            'rows_affected': result['rows_affected'],
                            'error': result['error']
                        })
                    if tahunan:
                        query = """
                        UPDATE v_pegawai_data_akhir_tahun_2025
                        SET peg_jabatan = null
                        WHERE peg_id = :peg_id
                        """
                        result = execute_update(query, {'peg_id': peg_id}, engine, f"Update vpd_tahunan for {nip_bawahan}")
                        log.append({
                            'nip_bawahan': nip_bawahan,
                            'nip_atasan': nip_atasan,
                            'table': 'v_pegawai_data_akhir_tahun_2025',
                            'status': 'success' if result['success'] else 'failed',
                            'rows_affected': result['rows_affected'],
                            'error': result['error']
                        })
                elif not peg_tugas_tambahan.empty:
                    peg_id = peg_tugas_tambahan['peg_id'].iloc[0]
                    if tw4:
                        query = """
                        UPDATE v_pegawai_data_tw4_2025
                        SET peg_tugas_tambahan = null
                        WHERE peg_id = :peg_id
                        """
                        result = execute_update(query, {'peg_id': peg_id}, engine, f"Update vpd_tw4 for {nip_bawahan}")
                        log.append({
                            'nip_bawahan': nip_bawahan,
                            'nip_atasan': nip_atasan,
                            'table': 'v_pegawai_data_tw4_2025',
                            'status': 'success' if result['success'] else 'failed',
                            'rows_affected': result['rows_affected'],
                            'error': result['error']
                        })
                    if tahunan:
                        query = """
                        UPDATE v_pegawai_data_akhir_tahun_2025
                        SET peg_tugas_tambahan = null
                        WHERE peg_id = :peg_id
                        """
                        result = execute_update(query, {'peg_id': peg_id}, engine, f"Update vpd_tahunan for {nip_bawahan}")
                        log.append({
                            'nip_bawahan': nip_bawahan,
                            'nip_atasan': nip_atasan,
                            'table': 'v_pegawai_data_akhir_tahun_2025',
                            'status': 'success' if result['success'] else 'failed',
                            'rows_affected': result['rows_affected'],
                            'error': result['error']
                        })
                else:
                    log.append({
                        'nip_bawahan': nip_bawahan,
                        'nip_atasan': nip_atasan,
                        'table': 'lookup',
                        'status': 'failed',
                        'rows_affected': 0,
                        'error': f'Jabatan ID {jabatan_id} not found'
                    })
                continue
            except Exception as e:
                log.append({
                    'nip_bawahan': nip_bawahan,
                    'nip_atasan': nip_atasan,
                    'table': 'lookup',
                    'status': 'failed',
                    'rows_affected': 0,
                    'error': f'Error parsing PLT: {str(e)}'
                })
                continue
                
        # If atasan PLT
        if 'PLT' in str(nip_atasan):
            # Get jabatan_id from 'PLT-[jabatan_id]'
            try:
                jabatan_id = nip_atasan.split('-')[1]
                
                query = """
                SELECT jabatan_nama FROM m_spg_jabatan
                WHERE jabatan_id = :jabatan_id
                """
                jabatan_result = pd.read_sql(text(query), engine, params={'jabatan_id': jabatan_id})
                
                if jabatan_result.empty:
                    log.append({
                        'nip_bawahan': nip_bawahan,
                        'nip_atasan': nip_atasan,
                        'table': 'lookup',
                        'status': 'failed',
                        'rows_affected': 0,
                        'error': f'Jabatan ID {jabatan_id} not found'
                    })
                    continue
                
                jabatan_nama = jabatan_result['jabatan_nama'].iloc[0]
                
            except Exception as e:
                log.append({
                    'nip_bawahan': nip_bawahan,
                    'nip_atasan': nip_atasan,
                    'table': 'parse',
                    'status': 'failed',
                    'rows_affected': 0,
                    'error': f'Error parsing PLT: {str(e)}'
                })
                continue
        else:
            # Use nip_atasan as is and look up nama_atasan
            try:
                query = """
                SELECT peg_nama FROM m_pegawai
                WHERE nip = :nip_atasan
                """
                nama_result = pd.read_sql(text(query), engine, params={'nip_atasan': nip_atasan})
                
                if nama_result.empty:
                    log.append({
                        'nip_bawahan': nip_bawahan,
                        'nip_atasan': nip_atasan,
                        'table': 'lookup',
                        'status': 'failed',
                        'rows_affected': 0,
                        'error': f'NIP atasan {nip_atasan} not found'
                    })
                    continue
                
                jabatan_nama = nama_result['peg_nama'].iloc[0]
                
            except Exception as e:
                log.append({
                    'nip_bawahan': nip_bawahan,
                    'nip_atasan': nip_atasan,
                    'table': 'lookup',
                    'status': 'failed',
                    'rows_affected': 0,
                    'error': f'Error looking up nama: {str(e)}'
                })
                continue

        # Update tahunan if requested
        if tahunan:
            query = """
            UPDATE v_pegawai_data_akhir_tahun_2025
            SET nip_atasan = :nip_atasan,
                nama_atasan = :nama_atasan
            WHERE peg_nip = :nip_bawahan
            """
            
            result = execute_update(
                query,
                {
                    'nip_atasan': nip_atasan,
                    'nama_atasan': jabatan_nama,
                    'nip_bawahan': nip_bawahan
                },
                engine,
                f"Update tahunan for {nip_bawahan}"
            )
            
            log.append({
                'nip_bawahan': nip_bawahan,
                'nip_atasan': nip_atasan,
                'nama_atasan': jabatan_nama,
                'table': 'v_pegawai_data_akhir_tahun_2025',
                'status': 'success' if result['success'] else 'failed',
                'rows_affected': result['rows_affected'],
                'error': result['error']
            })

        # Update tw4 if requested
        if tw4:
            query = """
            UPDATE v_pegawai_data_tw4_2025
            SET nip_atasan = :nip_atasan,
                nama_atasan = :nama_atasan
            WHERE peg_nip = :nip_bawahan
            """
            
            result = execute_update(
                query,
                {
                    'nip_atasan': nip_atasan,
                    'nama_atasan': jabatan_nama,
                    'nip_bawahan': nip_bawahan
                },
                engine,
                f"Update tw4 for {nip_bawahan}"
            )
            
            log.append({
                'nip_bawahan': nip_bawahan,
                'nip_atasan': nip_atasan,
                'nama_atasan': jabatan_nama,
                'table': 'v_pegawai_data_tw4_2025',
                'status': 'success' if result['success'] else 'failed',
                'rows_affected': result['rows_affected'],
                'error': result['error']
            })
    
    # Convert log to DataFrame
    df_log = pd.DataFrame(log)
    
    # Print summary
    print(f"\n{'='*60}")
    print(f"UPDATE SUMMARY")
    print(f"{'='*60}")
    print(f"Total operations: {len(df_log)}")
    print(f"Successful: {len(df_log[df_log['status'] == 'success'])}")
    print(f"Failed: {len(df_log[df_log['status'] == 'failed'])}")
    print(f"Total rows affected: {df_log['rows_affected'].sum()}")
    print(f"{'='*60}\n")
    
    if len(df_log[df_log['status'] == 'failed']) > 0:
        print("Failed operations:")
        print(df_log[df_log['status'] == 'failed'][['nip_bawahan', 'table', 'error']])
    
    return df_log

print("Atasan validation function with logging defined successfully!")

Atasan validation function with logging defined successfully!


In [ ]:
df_log = pd.DataFrame()

df_log = atasan_validasi_skp(df_validasi_atasan, df_vpd_tw4, df_vpd_tahunan)